In [1]:
import cv2
import mediapipe as mp
import os
import matplotlib.pyplot as plt
import time
import numpy as np

In [2]:
mp_holistic =mp.solutions.holistic 
mp_drawing= mp.solutions.drawing_utils

In [3]:
def mediapipe_detection(image,model):
    image= cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results= model.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image,results

In [4]:
mp.solutions.face_mesh.FACEMESH_TESSELATION

frozenset({(18, 17),
           (82, 38),
           (8, 9),
           (456, 248),
           (167, 2),
           (303, 271),
           (69, 104),
           (253, 450),
           (41, 42),
           (315, 16),
           (73, 74),
           (339, 373),
           (258, 385),
           (105, 63),
           (219, 48),
           (304, 272),
           (236, 217),
           (231, 22),
           (85, 84),
           (365, 367),
           (237, 218),
           (243, 112),
           (436, 427),
           (409, 410),
           (248, 456),
           (330, 266),
           (136, 135),
           (88, 95),
           (394, 430),
           (107, 108),
           (118, 117),
           (439, 438),
           (442, 443),
           (260, 466),
           (115, 48),
           (342, 276),
           (121, 128),
           (200, 421),
           (113, 124),
           (132, 137),
           (23, 230),
           (343, 357),
           (443, 444),
           (62, 78),
           (26,

In [5]:
def draw_landmarks(image, results):
    # Draw face landmarks using Face Mesh connections
    if results.face_landmarks:
        mp_drawing.draw_landmarks(
            image, 
            results.face_landmarks, 
            mp.solutions.face_mesh.FACEMESH_TESSELATION,
            mp_drawing.DrawingSpec(color=(80, 110, 10), thickness=1, circle_radius=1),  # Landmark dots
            mp_drawing.DrawingSpec(color=(80, 256, 121), thickness=1)  # Connections
        )
    
    # Draw pose landmarks
    if results.pose_landmarks:
        mp_drawing.draw_landmarks(
            image, 
            results.pose_landmarks, 
            mp_holistic.POSE_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=2, circle_radius=2),  # Landmark dots
            mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2)  # Connections
        )
    
    # Draw left-hand landmarks
    if results.left_hand_landmarks:
        mp_drawing.draw_landmarks(
            image, 
            results.left_hand_landmarks, 
            mp_holistic.HAND_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(121, 22, 76), thickness=2, circle_radius=2),  # Landmark dots
            mp_drawing.DrawingSpec(color=(121, 44, 250), thickness=2)  # Connections
        )
    
    # Draw right-hand landmarks
    if results.right_hand_landmarks:
        mp_drawing.draw_landmarks(
            image, 
            results.right_hand_landmarks, 
            mp_holistic.HAND_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2),  # Landmark dots
            mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=2)  # Connections
        )


In [6]:
cam = cv2.VideoCapture(0)

with mp_holistic.Holistic(min_detection_confidence=0.5,min_tracking_confidence=0.5) as holistic:
    while cam.isOpened():
        ret,frame= cam.read()
        image,results = mediapipe_detection(frame,holistic)
        
        draw_landmarks(image,results)
       
        cv2.imshow("Camera",image)
        if cv2.waitKey(10) == ord('q'):
            break
    cam.release()
    cv2.destroyAllWindows()

In [7]:
len(results.pose_landmarks.landmark)

33

In [8]:
def extract_keypoints(results):
    pose = np.array([[res.x,res.y,res.z,res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(132*3)
    lh = np.array([[res.x,res.y,res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x,res.y,res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    face = np.array([[res.x,res.y,res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(1404)
    return np.concatenate([pose,face,lh,rh])

In [9]:
extract_keypoints(results).shape

(1662,)

In [10]:
# path for the exported data
DATA_PATH = os.path.join('MP_Data')

# Actions we will try to detect
actions = np.array(['Hello','Thanks','I Love You'])

# No of Videos
no_videos = 30

# no of Sequences
no_sequences = 30


In [11]:


# Create directories for storing data
for action in actions:
    for seq in range(no_videos):
        # Define the path for the action and sequence
        dir_path = os.path.join(DATA_PATH, action, str(seq))
        try:
            os.makedirs(dir_path, exist_ok=True)  # Use exist_ok to avoid exceptions if the directory exists
        except Exception as e:
            print(f"Error creating directory {dir_path}: {e}")


In [12]:
cam = cv2.VideoCapture(0)

with mp_holistic.Holistic(min_detection_confidence=0.5,min_tracking_confidence=0.5) as holistic:
    for action in actions:
        for sequence in range(no_videos):
            for frame_num in range(no_sequences):
                
                ret,frame= cam.read()
                image,results = mediapipe_detection(frame,holistic)
        
                draw_landmarks(image,results)
               
                
                if frame_num == 0: 
                    cv2.putText(image, 'STARTING COLLECTION', (120,200), 
                               cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255, 0), 4, cv2.LINE_AA)
                    cv2.putText(image, 'Collecting frames for {} Video Number {}'.format(action, sequence), (15,12), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
                    # Show to screen
                    cv2.imshow('OpenCV Feed', image)
                    cv2.waitKey(2000)
                else: 
                    cv2.putText(image, 'Collecting frames for {} Video Number {}'.format(action, sequence), (15,12), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
                    # Show to screen
                    cv2.imshow('OpenCV Feed', image)
                    
                    
                keypoints = extract_keypoints(results)
                npy_path = os.path.join(DATA_PATH,action, str(sequence), str(frame_num))
                np.save(npy_path,keypoints)
                
                if cv2.waitKey(10) == ord('q'):
                    break
                    
    cam.release()
    cv2.destroyAllWindows()

IndentationError: unexpected indent (2447370183.py, line 2)

In [14]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

In [15]:
label_map = {label:num for num, label in enumerate(actions)}

In [16]:
label_map

{'Hello': 0, 'Thanks': 1, 'I Love You': 2}

In [17]:
sequences,labels =[],[]
for action in actions:
    for sequence in np.array(os.listdir(os.path.join(DATA_PATH,action))).astype(int):
        window=[]
        for frame_num in range(no_sequences):
            res= np.load(os.path.join(DATA_PATH,action, str(sequence), "{}.npy".format(frame_num)))
            window.append(res)
        sequences.append(window)
        labels.append(label_map[action])

In [18]:
X = np.array(sequences)
y = to_categorical(labels).astype(int)

In [19]:
X.shape

(90, 30, 1662)

In [20]:
y.shape

(90, 3)

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05)

In [22]:
y_train.shape

(85, 3)

In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM
from tensorflow.keras.callbacks import TensorBoard


In [24]:
import os

log_dir = os.path.join('Logs')
tb_callback = TensorBoard(log_dir=log_dir)

In [28]:
# Define the model
model = Sequential()
model.add(LSTM(64, return_sequences=True, activation='relu', input_shape=(30, 1662)))
model.add(LSTM(128, return_sequences=True, activation='relu'))
model.add(LSTM(64, return_sequences=False, activation='relu'))
model.add(Dense(64, activation='relu'))  # Fixed typo
model.add(Dense(32, activation='relu'))  # Corrected activation
model.add(Dense(actions.shape[0], activation='softmax'))  # Fixed parentheses

In [29]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])




In [30]:
# Example training code (replace X_train, y_train, etc., with your dataset)
model.fit(X_train, y_train, epochs=190, callbacks=[tb_callback])

Epoch 1/190
3/3 [==============================] - 11s 122ms/step - loss: 14.4404 - categorical_accuracy: 0.3882
Epoch 2/190
3/3 [==============================] - 0s 114ms/step - loss: 24.8310 - categorical_accuracy: 0.3529
Epoch 3/190
3/3 [==============================] - 0s 111ms/step - loss: 18.2786 - categorical_accuracy: 0.3412
Epoch 4/190
3/3 [==============================] - 0s 113ms/step - loss: 33.3729 - categorical_accuracy: 0.3059
Epoch 5/190
3/3 [==============================] - 0s 118ms/step - loss: 17.4906 - categorical_accuracy: 0.4000
Epoch 6/190
3/3 [==============================] - 0s 115ms/step - loss: 3.5119 - categorical_accuracy: 0.2824
Epoch 7/190
3/3 [==============================] - 0s 107ms/step - loss: 2.4768 - categorical_accuracy: 0.2706
Epoch 8/190
3/3 [==============================] - 0s 106ms/step - loss: 1.4884 - categorical_accuracy: 0.3059
Epoch 9/190
3/3 [==============================] - 0s 110ms/step - loss: 1.2176 - categorical_accuracy: 0.

3/3 [==============================] - 0s 108ms/step - loss: 0.2693 - categorical_accuracy: 0.8941
Epoch 75/190
3/3 [==============================] - 0s 121ms/step - loss: 0.3054 - categorical_accuracy: 0.8588
Epoch 76/190
3/3 [==============================] - 0s 109ms/step - loss: 0.2302 - categorical_accuracy: 0.9294
Epoch 77/190
3/3 [==============================] - 0s 108ms/step - loss: 0.2487 - categorical_accuracy: 0.9059
Epoch 78/190
3/3 [==============================] - 0s 113ms/step - loss: 0.2932 - categorical_accuracy: 0.8588
Epoch 79/190
3/3 [==============================] - 0s 107ms/step - loss: 0.2945 - categorical_accuracy: 0.8824
Epoch 80/190
3/3 [==============================] - 0s 108ms/step - loss: 0.5215 - categorical_accuracy: 0.8118
Epoch 81/190
3/3 [==============================] - 0s 115ms/step - loss: 0.3671 - categorical_accuracy: 0.8824
Epoch 82/190
3/3 [==============================] - 0s 109ms/step - loss: 0.3272 - categorical_accuracy: 0.9176
Epoch

3/3 [==============================] - 0s 107ms/step - loss: 0.2764 - categorical_accuracy: 0.9294
Epoch 147/190
3/3 [==============================] - 0s 106ms/step - loss: 0.2696 - categorical_accuracy: 0.8706
Epoch 148/190
3/3 [==============================] - 0s 113ms/step - loss: 0.2082 - categorical_accuracy: 0.9294
Epoch 149/190
3/3 [==============================] - 0s 111ms/step - loss: 0.2389 - categorical_accuracy: 0.9176
Epoch 150/190
3/3 [==============================] - 0s 110ms/step - loss: 0.6839 - categorical_accuracy: 0.8118
Epoch 151/190
3/3 [==============================] - 0s 105ms/step - loss: 0.7743 - categorical_accuracy: 0.7176
Epoch 152/190
3/3 [==============================] - 0s 105ms/step - loss: 0.4638 - categorical_accuracy: 0.7882
Epoch 153/190
3/3 [==============================] - 0s 115ms/step - loss: 0.2751 - categorical_accuracy: 0.9176
Epoch 154/190
3/3 [==============================] - 0s 104ms/step - loss: 0.3014 - categorical_accuracy: 0.94

In [31]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_3 (LSTM)               (None, 30, 64)            442112    
                                                                 
 lstm_4 (LSTM)               (None, 30, 128)           98816     
                                                                 
 lstm_5 (LSTM)               (None, 64)                49408     
                                                                 
 dense_3 (Dense)             (None, 64)                4160      
                                                                 
 dense_4 (Dense)             (None, 32)                2080      
                                                                 
 dense_5 (Dense)             (None, 3)                 99        
                                                                 
Total params: 596675 (2.28 MB)
Trainable params: 59667

In [49]:
res=model.predict(X_test)
res[0]

1/1 [==============================] - 0s 58ms/step


array([1.3676303e-23, 1.0000000e+00, 2.1396772e-29], dtype=float32)

In [44]:
actions[np.argmax(res[3])]

'Thanks'

In [45]:
actions[np.argmax(y_test[3])]

'Thanks'

In [47]:
# Save the model in the recommended .keras format
model.save('my_model.keras')


In [55]:
import tensorflow as tf
model = tf.keras.models.load_model('my_model.keras')

In [56]:
colors = [(245,117,16), (117,245,16), (16,117,245)]
def prob_viz(res, actions, input_frame, colors):
    output_frame = input_frame.copy()
    for num, prob in enumerate(res):
        cv2.rectangle(output_frame, (0,60+num*40), (int(prob*100), 90+num*40), colors[num], -1)
        cv2.putText(output_frame, actions[num], (0, 85+num*40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2, cv2.LINE_AA)
        
    return output_frame

In [58]:
sequence = []
sentence = []
predictions = []
threshold = 0.5

cap = cv2.VideoCapture(0)
# Set mediapipe model 
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():

        # Read feed
        ret, frame = cap.read()

        # Make detections
        image, results = mediapipe_detection(frame, holistic)
        print(results)
        
        # Draw landmarks
        draw_landmarks(image, results)
        
        # 2. Prediction logic
        keypoints = extract_keypoints(results)
        sequence.append(keypoints)
        sequence = sequence[-30:]
        
        if len(sequence) == 30:
            res = model.predict(np.expand_dims(sequence, axis=0))[0]
            print(actions[np.argmax(res)])
            predictions.append(np.argmax(res))
            
            
        #3. Viz logic
            if np.unique(predictions[-10:])[0]==np.argmax(res): 
                if res[np.argmax(res)] > threshold: 
                    
                    if len(sentence) > 0: 
                        if actions[np.argmax(res)] != sentence[-1]:
                            sentence.append(actions[np.argmax(res)])
                    else:
                        sentence.append(actions[np.argmax(res)])

            if len(sentence) > 5: 
                sentence = sentence[-5:]

            # Viz probabilities
            image = prob_viz(res, actions, image, colors)
            
        cv2.rectangle(image, (0,0), (640, 40), (245, 117, 16), -1)
        cv2.putText(image, ' '.join(sentence), (3,30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
        
        # Show to screen
        cv2.imshow('OpenCV Feed', image)

        # Break gracefully
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()



<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.soluti

1/1 [==============================] - 0s 37ms/step
Hello
<class 'mediapipe.python.solution_base.SolutionOutputs'>
1/1 [==============================] - 0s 36ms/step
Hello
<class 'mediapipe.python.solution_base.SolutionOutputs'>
1/1 [==============================] - 0s 38ms/step
Hello
<class 'mediapipe.python.solution_base.SolutionOutputs'>
1/1 [==============================] - 0s 38ms/step
Hello
<class 'mediapipe.python.solution_base.SolutionOutputs'>
1/1 [==============================] - 0s 36ms/step
Hello
<class 'mediapipe.python.solution_base.SolutionOutputs'>
1/1 [==============================] - 0s 38ms/step
Hello
<class 'mediapipe.python.solution_base.SolutionOutputs'>
1/1 [==============================] - 0s 41ms/step
Hello
<class 'mediapipe.python.solution_base.SolutionOutputs'>
1/1 [==============================] - 0s 35ms/step
Hello
<class 'mediapipe.python.solution_base.SolutionOutputs'>
1/1 [==============================] - 0s 38ms/step
Hello
<class 'mediapipe.pyth

1/1 [==============================] - 0s 36ms/step
Thanks
<class 'mediapipe.python.solution_base.SolutionOutputs'>
1/1 [==============================] - 0s 39ms/step
Thanks
<class 'mediapipe.python.solution_base.SolutionOutputs'>
1/1 [==============================] - 0s 40ms/step
Thanks
<class 'mediapipe.python.solution_base.SolutionOutputs'>
1/1 [==============================] - 0s 36ms/step
Thanks
<class 'mediapipe.python.solution_base.SolutionOutputs'>
1/1 [==============================] - 0s 35ms/step
Thanks
<class 'mediapipe.python.solution_base.SolutionOutputs'>
1/1 [==============================] - 0s 36ms/step
Thanks
<class 'mediapipe.python.solution_base.SolutionOutputs'>
1/1 [==============================] - 0s 35ms/step
Thanks
<class 'mediapipe.python.solution_base.SolutionOutputs'>
1/1 [==============================] - 0s 39ms/step
Thanks
<class 'mediapipe.python.solution_base.SolutionOutputs'>
1/1 [==============================] - 0s 37ms/step
Thanks
<class 'media